In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from articulate import articulate
from omegaconf import OmegaConf
from dotenv import load_dotenv
from articulate_anything.utils.viz import (
    show_video, 
    display_code, 
    show_videos, 
    display_codes,
    show_images,
    get_frames_from_video,
)
from articulate_anything.utils.utils import load_config, join_path
from articulate_anything.utils.cotracker_utils import make_cotracker
from PIL import Image
import json

In [ ]:
import os
os.chdir("..")

In [ ]:
API_KEY = "YOUR-ACTUAL-API-KEY"
## we have our API key stored in a .env file
## Comment the `load_dotenv` and `os.environ.get` lines if you just want to use
## your API key directly
load_dotenv()
API_KEY = os.environ.get('API_KEY')

task = "suitcase"

Let's first take a look at the ground-truth video

In [ ]:
video_path = f"datasets/in-the-wild-dataset/videos/{task}.mp4"

In [ ]:
show_video(video_path)

Let's extract the first frame from the video and use that as the image prompt

In [ ]:
frames = get_frames_from_video(video_path,num_frames=5,)
frames[0]

In [ ]:
if not os.path.exists("datasets/in-the-wild-dataset/images"):
    os.makedirs("datasets/in-the-wild-dataset/images")
image_path = f"datasets/in-the-wild-dataset/images/{task}.png"
frames[0].save(image_path)

We can annotate the motion in the video using Cotracker

In [ ]:
cfg = load_config()

Before starting articulation, please make sure that the dataset is preprocessed by running

   ```bash
   python articulate_anything/preprocess/preprocess_partnet.py parallel={int} modality={image}
   ```
This renders a front-view image for each object in the PartNet-Mobility dataset. This is necessary for our mesh retrieval as we will compare the visual similarity between the input image or video against each rendered template object.

In [ ]:
modality = "image"
prompt = image_path

Note: currently Gemini does not support multi-modal prompt finetuning. We have to use few-shot prompting instead. It's important to pick the examples with some care for your use case.

In [ ]:
cfg = load_config()
cfg.prompt = prompt
cfg.modality = modality
cfg.out_dir = join_path("results", modality, task)



use_cotracker = True # {True, False}
mode = "image"
actor_prompting_type = "basic" # {basic, incontext}
critic_prompting_type = "incontext" # {basic, incontext}

cfg.joint_actor.mode = mode
cfg.joint_actor.use_cotracker = use_cotracker
cfg.joint_actor.type = actor_prompting_type
cfg.joint_actor.targetted_affordance = False

cfg.joint_actor.examples_dir = "datasets/multi_modal_incontext_examples/joint_actor/in_context_actor_examples_datasets" ## Put your examples here


cfg.joint_critic.mode = mode
cfg.joint_critic.use_cotracker = use_cotracker
cfg.joint_critic.type = critic_prompting_type

cfg.joint_critic.examples_dir = "datasets/multi_modal_incontext_examples/joint_critic/in_context_examples_datasets" ## Put your examples here

cfg.actor_critic.actor_only = True

## important to set correctly for the joint_critic to works properly
## this should have the same direction as the ground-truth video
cfg.simulator.flip_video = False ## flip time for suitcase
cfg.simulator.ray_tracing=False
cfg.simulator.floor_texture = "plain"


# cfg.category_selector.topk = 3
cfg.category_selector.topk = 1 ## how many top categories should we search for an object template match
# this is because PartNet-Mobility dataset categories labels are sparse and sometimes not great


cfg.obj_selector.frame_index = 0

cfg.actor_critic.max_iter = 2
cfg.model_name = "gemini-1.5-flash-latest"


cfg.api_key = API_KEY ## loaded from .env file

In [ ]:
steps = articulate(cfg);

## Mesh Retrieval

Let's inspect the steps starting with the mesh retrieval

In [ ]:
mesh_retrieval = steps["Mesh Retrieval"]

In [ ]:
mesh_retrieval["Category Selection"].load_prediction()

In [ ]:
obj_selector = mesh_retrieval["Object Selection"]
obj_selector.load_prediction()

In [ ]:
obj_selector.load_predicted_rendering()

## Link Placement

In [ ]:
link_art = steps["Link Articulation"]
link_actors = link_art["Link actor"]
link_critics = link_art["Link critic"]
assert len(link_actors) == len(link_critics)
print(f"Link placement runs for {len(link_actors)} iteration(s)")

In [ ]:
link_codes = [link_actor.load_prediction() for link_actor in link_actors]

link_preds = [link_actor.load_predicted_rendering() for link_actor in link_actors]

In [ ]:
show_images(link_preds)
display_codes(link_codes)

Note: 
1. some object part names need relabeling because PartNet-Mobility part labeling is not great. For example, a lid's pump rod would be labeled as `lid` or a door knob would be labeled as `door`. **solution**: gemini annotation based on mesh images + maybe some manual relabeling. This is done automatically by the `preprocess_partnet.py` script.
2. Some meshes are also not "clean" so there would be floating points in the air that affect our collision detection algorithm  and thus our link placement but might not immediate visible to the eye. **solution**: use [open3d](https://www.open3d.org/docs/release/tutorial/geometry/mesh.html#Connected-components) to clean up the meshes.

## Joint Prediction

In [ ]:
joint_art = steps["Joint Articulation"]
joint_actors = joint_art["Joint actor"]


print(f"Joint prediction runs for {len(joint_actors)} iteration(s)")

In [ ]:
joint_codes = [joint_actor.load_prediction() for joint_actor in joint_actors]
joint_preds = [joint_actor.load_predicted_rendering() for joint_actor in joint_actors]


In [ ]:
show_videos(joint_preds, width=512, height=512)
display_codes(joint_codes)